In [6]:
#@title 📦 Celda 1: SETUP (ejecutar primero, solo una vez)
#@markdown Instala ComfyUI, wrapper, rasterizer y descarga todos los modelos.

import subprocess, os, shutil

def run(cmd, desc=""):
    print(f"\n{'='*50}\n📦 {desc}\n{'='*50}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout:
        for l in [l for l in r.stdout.strip().split('\n') if l.strip()][-3:]:
            print(l)
    if r.returncode != 0:
        for l in [l for l in r.stderr.strip().split('\n') if 'error' in l.lower()][-3:]:
            print(f"⚠️ {l}")
    return r.returncode == 0

import torch
assert torch.cuda.is_available(), "❌ No GPU! Runtime → Change runtime type → T4 GPU"
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

# ComfyUI
if not os.path.exists('/content/ComfyUI'):
    run("git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI", "Cloning ComfyUI")
os.chdir('/content/ComfyUI')
run("pip install -q -r requirements.txt", "Installing ComfyUI")
print("✅ ComfyUI")

# Wrapper
wrapper = '/content/ComfyUI/custom_nodes/ComfyUI-Hunyuan3DWrapper'
if not os.path.exists(wrapper):
    run(f"git clone https://github.com/kijai/ComfyUI-Hunyuan3DWrapper.git {wrapper}", "Cloning wrapper")
run(f"pip install -q -r {wrapper}/requirements.txt", "Installing wrapper deps")
run("pip install -q pyfqmr", "Installing pyfqmr (mesh simplification)")
print("✅ Wrapper")

# Rasterizer
rasterizer_dir = f"{wrapper}/hy3dgen/texgen/custom_rasterizer"
if os.path.exists(rasterizer_dir):
    run(f"export PATH=/usr/local/cuda/bin:$PATH && pip install {rasterizer_dir}", "Compiling rasterizer")
print("✅ Rasterizer")

# Cloudflared
run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1 && rm cloudflared-linux-amd64.deb", "Cloudflared")
print("✅ Cloudflared")

# Models
from huggingface_hub import hf_hub_download, snapshot_download
models = '/content/ComfyUI/models'

def dl(desc, func, *args, **kwargs):
    print(f"\n📥 {desc}...")
    func(*args, **kwargs)
    print(f"✅ {desc}")

# DiT
dit = f'{models}/checkpoints/hunyuan3d-dit-v2-0-fast.safetensors'
if not os.path.exists(dit):
    dl("DiT (~4.9 GB)", hf_hub_download, "tencent/Hunyuan3D-2", "hunyuan3d-dit-v2-0-fast/model.fp16.safetensors", local_dir="/content/tmp")
    shutil.move("/content/tmp/hunyuan3d-dit-v2-0-fast/model.fp16.safetensors", dit)
dit_w = f'{models}/diffusion_models/hunyuan3d-dit-v2-0-fast.safetensors'
if not os.path.exists(dit_w):
    shutil.copy2(dit, dit_w)

# VAE
vae = f'{models}/vae/hunyuan3d-vae-v2-0.safetensors'
if not os.path.exists(vae):
    dl("VAE (~430 MB)", hf_hub_download, "tencent/Hunyuan3D-2", "hunyuan3d-vae-v2-0/model.fp16.safetensors", local_dir="/content/tmp")
    shutil.move("/content/tmp/hunyuan3d-vae-v2-0/model.fp16.safetensors", vae)

# SDXL
sdxl = f'{models}/checkpoints/sd_xl_base_1.0.safetensors'
if not os.path.exists(sdxl):
    dl("SDXL (~6.5 GB)", hf_hub_download, "stabilityai/stable-diffusion-xl-base-1.0", "sd_xl_base_1.0.safetensors", local_dir=f"{models}/checkpoints")

# Paint
paint = f'{models}/diffusers/hunyuan3d-paint-v2-0/unet/diffusion_pytorch_model.safetensors'
if not os.path.exists(paint):
    dl("Paint (~8.6 GB)", snapshot_download, "tencent/Hunyuan3D-2", allow_patterns=["hunyuan3d-paint-v2-0/*"], local_dir="/content/tmp")
    dst = f'{models}/diffusers/hunyuan3d-paint-v2-0'
    os.makedirs(dst, exist_ok=True)
    shutil.copytree("/content/tmp/hunyuan3d-paint-v2-0", dst, dirs_exist_ok=True)

# Delight
delight = f'{models}/diffusers/hunyuan3d-delight-v2-0/unet/diffusion_pytorch_model.safetensors'
if not os.path.exists(delight):
    dl("Delight (~4 GB)", snapshot_download, "tencent/Hunyuan3D-2", allow_patterns=["hunyuan3d-delight-v2-0/*"], local_dir="/content/tmp")
    dst = f'{models}/diffusers/hunyuan3d-delight-v2-0'
    os.makedirs(dst, exist_ok=True)
    shutil.copytree("/content/tmp/hunyuan3d-delight-v2-0", dst, dirs_exist_ok=True)

# Cleanup
subprocess.run("rm -rf /content/tmp", shell=True)
import gc; gc.collect()
subprocess.run("sync && echo 3 > /proc/sys/vm/drop_caches", shell=True)

print(f"\n{'🎉'*20}")
print("SETUP COMPLETO. Ahora ejecuta Celda 2.")
print(f"{'🎉'*20}")


✅ GPU: Tesla T4

📦 Installing ComfyUI
✅ ComfyUI

📦 Installing wrapper deps

📦 Installing pyfqmr (mesh simplification)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 907.0/907.0 kB 27.5 MB/s eta 0:00:00
✅ Wrapper

📦 Compiling rasterizer
    Uninstalling custom_rasterizer-0.1.0+torch2100.cuda128:
      Successfully uninstalled custom_rasterizer-0.1.0+torch2100.cuda128
✅ Rasterizer

📦 Cloudflared
✅ Cloudflared

🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉
SETUP COMPLETO. Ahora ejecuta Celda 2.
🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉


In [3]:
#@title 🎨 Celda 2: GENERAR IMAGEN + FORMA 3D (ejecutar después del setup)
#@markdown Genera imagen con SDXL y modelo 3D con Hunyuan3D. NO incluye textura.
#@markdown Cuando termine, DETÉN esta celda (botón Stop) antes de ejecutar Celda 3.

import subprocess, threading, re, time, os, gc

# Free any leftover memory
gc.collect()
subprocess.run("sync && echo 3 > /proc/sys/vm/drop_caches", shell=True)

print("🚀 Iniciando ComfyUI (solo SDXL + Hunyuan3D shape)...")
print("   Modelos de paint NO se cargan en esta fase.\n")

# Tunnel
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

def show_url():
    for line in tunnel.stderr:
        match = re.search(r'https://[\w-]+\.trycloudflare\.com', line.decode())
        if match:
            print(f"\n{'🌐'*20}")
            print(f"\n  ✅ COMFYUI LISTO:")
            print(f"  👉 {match.group(0)}")
            print(f"\n  1. Abre la URL en tu navegador")
            print(f"  2. Arrastra 'workflow_shape.json'")
            print(f"  3. Edita el prompt y dale Ejecutar")
            print(f"  4. Cuando tengas el mesh, DETÉN esta celda")
            print(f"  5. Luego ejecuta Celda 3 para texturizar")
            print(f"\n{'🌐'*20}\n")
            break

threading.Thread(target=show_url, daemon=True).start()
time.sleep(3)

os.chdir('/content/ComfyUI')
os.system("python main.py --listen 127.0.0.1 --port 8188 --lowvram --force-fp16")


🚀 Iniciando ComfyUI (solo SDXL + Hunyuan3D shape)...
   Modelos de paint NO se cargan en esta fase.


🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐

  ✅ COMFYUI LISTO:
  👉 https://fixes-bids-thinkpad-elect.trycloudflare.com

  1. Abre la URL en tu navegador
  2. Arrastra 'workflow_shape.json'
  3. Edita el prompt y dale Ejecutar
  4. Cuando tengas el mesh, DETÉN esta celda
  5. Luego ejecuta Celda 3 para texturizar

🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐🌐



2

In [ ]:
#@title 🎨 Celda 3: TEXTURIZAR MESH (ejecutar después de detener Celda 2)
#@markdown Carga el mesh generado en Celda 2 y le aplica textura con Paint.
#@markdown Asegúrate de haber DETENIDO la Celda 2 primero para liberar RAM.

import subprocess, threading, re, time, os, gc

# Install pyfqmr for mesh simplification
subprocess.run("pip install -q pyfqmr", shell=True)

# Free memory from previous ComfyUI instance
gc.collect()
subprocess.run("sync && echo 3 > /proc/sys/vm/drop_caches", shell=True)
time.sleep(2)

# Copy outputs from Celda 2 to input folder so LoadImage/LoadMesh can find them
import glob, shutil
os.makedirs('/content/ComfyUI/input', exist_ok=True)
for f in glob.glob('/content/ComfyUI/output/generated/*.png'):
    shutil.copy2(f, '/content/ComfyUI/input/')
    print(f"📋 Copied {os.path.basename(f)} to input/")
for f in glob.glob('/content/ComfyUI/output/mesh/*.glb'):
    shutil.copy2(f, '/content/ComfyUI/input/')
    print(f"📋 Copied {os.path.basename(f)} to input/")

print("\n🚀 Iniciando ComfyUI (solo Paint + Delight)...")
print("   SDXL y DiT NO se cargan en esta fase.\n")

# Tunnel
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

def show_url():
    for line in tunnel.stderr:
        match = re.search(r'https://[\w-]+\.trycloudflare\.com', line.decode())
        if match:
            print(f"\n{'🎨'*20}")
            print(f"\n  ✅ COMFYUI LISTO (PAINT):")
            print(f"  👉 {match.group(0)}")
            print(f"\n  1. Abre la URL en tu navegador")
            print(f"  2. Arrastra 'workflow_paint.json'")
            print(f"  3. Selecciona tu mesh e imagen en los nodos Load")
            print(f"  4. Dale Ejecutar")
            print(f"\n{'🎨'*20}\n")
            break

threading.Thread(target=show_url, daemon=True).start()
time.sleep(3)

os.chdir('/content/ComfyUI')
os.system("python main.py --listen 127.0.0.1 --port 8188 --lowvram --force-fp16")


📋 Copied imagen_00004_.png to input/
📋 Copied imagen_00002_.png to input/
📋 Copied imagen_00001_.png to input/
📋 Copied imagen_00003_.png to input/
📋 Copied sin_textura_00002_.glb to input/
📋 Copied sin_textura_00001_.glb to input/
📋 Copied con_textura_00001_.glb to input/
📋 Copied sin_textura_00003_.glb to input/
📋 Copied modelo_00001_.glb to input/

🚀 Iniciando ComfyUI (solo Paint + Delight)...
   SDXL y DiT NO se cargan en esta fase.


🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨

  ✅ COMFYUI LISTO (PAINT):
  👉 https://results-joe-joins-purse.trycloudflare.com

  1. Abre la URL en tu navegador
  2. Arrastra 'workflow_paint.json'
  3. Selecciona tu mesh e imagen en los nodos Load
  4. Dale Ejecutar

🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨🎨

